## Tokenizing Text

In [1]:
with open('the-verdict.txt', 'r', encoding='utf-8') as f:
    story = f.read()
print(f'total number of characters: {len(story)}')
print(story[:100])

total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [2]:
import re

正则匹配符中的 () 是捕获组，加了后会保留分隔符

In [4]:
text = "Hello, world. This is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test.']


In [4]:
result1 = re.split(r'\s', text)
print(result1)

['Hello,', 'world.', 'This', 'is', 'a', 'test.']


在如果遇到两个连续的分隔符，或者分隔符在开头、结尾，那么就会产生 '' 这样的空字符

In [6]:
re1 = re.split(r'([,.]|\s)', text)
print(re1)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [7]:
re2 = re.split(r'[,.]|\s', text)
print(re2)

['Hello', '', 'world', '', 'This', 'is', 'a', 'test', '']


In [8]:
# 排除空白

re1_stripe = [item for item in re1 if item.strip()]
print(re1_stripe)

['Hello', ',', 'world', '.', 'This', 'is', 'a', 'test', '.']


In [18]:
text = "Hello, world. Is' t-- a test?"
re1 = re.split(r'([,.;:?!_"()\']|--|\s)', text)
print(re1)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'Is', "'", '', ' ', 't', '--', '', ' ', 'a', ' ', 'test', '?', '']


In [21]:
preprocessed = re.split(r'([,.;:?!_"()\']|--|\s)', story)
preprocessed = [item for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens into token IDs

In [22]:
all_words = set(sorted(set(preprocessed))) # 之所以两次set，因为后面可能多次进行extend同样的内容
vocab_size = len(all_words)
print(f'vocab size: {vocab_size}')

vocab size: 1130


In [24]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for token, integer in vocab.items():
    print(f'{token}: {integer}')
    if integer>20:
        break

!: 0
": 1
': 2
(: 3
): 4
,: 5
--: 6
.: 7
:: 8
;: 9
?: 10
A: 11
Ah: 12
Among: 13
And: 14
Are: 15
Arrt: 16
As: 17
At: 18
Be: 19
Begin: 20
Burlington: 21


decode方法里的 re.sub(r'\s+([,.?!"()\'])', r'\1', strs)

r'\1'是反向引用语言，表示前面的正则匹配式匹配到的第一个结果。这里只有一个捕获组()，因此指代的是匹配到的标点符号。
结合re.sub方法，表示把“空格+标点”替换为“标点”

In [25]:
class SimplerTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_token = {integer:token for token,integer in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.;:?!_"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()] # 有else，条件语句就要写在前面
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        strs = " ".join([self.int_to_token[id_] for id_ in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', strs)
        return text

In [28]:
tokenizer = SimplerTokenizer(vocab)
text = """
I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs .
"""
ids = tokenizer.encode(text)
print(ids)

[53, 244, 535, 67, 7, 37, 100, 6, 549, 602, 25, 897, 6, 326, 549, 1042, 116, 7, 1, 73, 297, 585, 2, 850, 498, 1016, 866, 988, 1059, 722, 697, 769, 2, 1083, 1051, 9, 239, 53, 359, 2, 970, 998, 722, 987, 5, 66, 7, 83, 6, 988, 646, 1016, 16, 584, 145, 53, 998, 722, 7, 1, 93, 1116, 5, 727, 67, 7]


In [29]:
print(tokenizer.decode(ids))
# 注意文本最后的空格没了

I can hear Mrs. Gideon Thwing -- his last Chicago sitter -- deploring his unaccountable abdication." Of course it' s going to send the value of my picture' way up ; but I don' t think of that, Mr. Rickham -- the loss to Arrt is all I think of." The word, on Mrs.


In [30]:
t1 = "Hello, do you like milk tea?"
ids = tokenizer.encode(t1)

KeyError: 'Hello'

## Adding special context tokens

In [38]:
all_words.extend(["<|unk|>", "<|endoftext|>"])
all_words_extended = list(set(all_words))
all_words_extended

['what',
 'stroke',
 'familiar',
 'negatived',
 'between',
 'fingers',
 'dim',
 'for',
 'fostered',
 'florid',
 'find',
 'one',
 'egregious',
 'appearance',
 'foreseen',
 'perfect',
 'stay',
 'terraces',
 'went',
 'craft',
 'flowers',
 'incense',
 'aside',
 'grayish',
 'have',
 'mediocrity',
 'prodigious',
 'friend',
 'Moon-dancers',
 'doesn',
 'failed',
 'swept',
 'amusing',
 'brings',
 'previous',
 'claimed',
 'became',
 'Croft',
 'secret',
 'threw',
 'sign',
 'heart',
 'abruptly',
 'could',
 'My',
 'so',
 'poised',
 'traps',
 'tribute',
 'surest',
 'must',
 'queerly',
 'haven',
 'three',
 'idea',
 'public',
 'straining',
 'trace',
 'laughed',
 'sunburn',
 'Grindle',
 'elbow',
 'women',
 'learned',
 'wouldn',
 'crumbled',
 'out',
 'travelled',
 'shrugged',
 'atom',
 'underlay',
 'thought',
 ',',
 'able',
 'hardly',
 'HAD',
 'pines',
 'ensuing',
 'turned',
 'marble',
 'axioms',
 'mirrors',
 'problem',
 'Dubarry',
 'get',
 'just',
 'nervousness',
 'platitudes',
 'presenting',
 'academi

In [43]:
all_words_extended.index("<|unk|>")

1093

In [44]:
class SimplerTokenizerNew:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_token = {integer:token for token,integer in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.;:?!_"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int.keys() else "<|unk|>" for item in preprocessed ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        strs = " ".join([self.int_to_token[id_] for id_ in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', strs)
        return text

In [46]:
vocab_new = {token:integer for integer,token in enumerate(all_words_extended)}

In [47]:

text = "Hello, do you like milk tea?"
tokenizer_new = SimplerTokenizerNew(vocab_new)
print(tokenizer_new.encode(text))
# Hello 被转换为 1093——"<|unk|>"

[1093, 72, 774, 177, 393, 1093, 258, 858]


In [48]:
tokenizer_new.decode(tokenizer_new.encode(text))

'<|unk|>, do you like <|unk|> tea?'

In [49]:
text1 = "Hello, do you like milk tea?<|endoftext|> I like eat some bananas after breakfast."
tokenizer_new.decode(tokenizer_new.encode(text1))

'<|unk|>, do you like <|unk|> tea? <|endoftext|> I like <|unk|> some <|unk|> after <|unk|>.'

## Byte pair encoding

In [2]:
import tiktoken
tokenizer =  tiktoken.get_encoding('gpt2')

In [3]:
text = "Hello, do you like milk tea? <|unk|> however, I don't think so."
ids = tokenizer.encode(text)
print(ids)

[15496, 11, 466, 345, 588, 7545, 8887, 30, 1279, 91, 2954, 91, 29, 2158, 11, 314, 836, 470, 892, 523, 13]


In [4]:
string = tokenizer.decode(ids)
print(string)

Hello, do you like milk tea? <|unk|> however, I don't think so.


In [5]:
tokenizer.encode('<|unk|>')

[27, 91, 2954, 91, 29]

## Data sampling with a sliding window

In [6]:
with open('the-verdict.txt', 'r', encoding='utf8') as f:
    text = f.read()

ids = tokenizer.encode(text)

In [7]:
len(ids)

5145

In [8]:
max_length = 4
print(f'x: {ids[:max_length]}')
print(f'y: {ids[1:max_length+1]}')

x: [40, 367, 2885, 1464]
y: [367, 2885, 1464, 1807]


In [12]:
for i in range(1, max_length+1):
    print(ids[:i], " --> ", ids[i])

[40]  -->  367
[40, 367]  -->  2885
[40, 367, 2885]  -->  1464
[40, 367, 2885, 1464]  -->  1807


In [14]:
for i in range(1, max_length+1):
    print(tokenizer.decode(ids[:i]), ' --> ', tokenizer.decode([ids[i]]))

I  -->   H
I H  -->  AD
I HAD  -->   always
I HAD always  -->   thought


In [1]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text)

        for i in range(0, len(token_ids)-max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i+max_length]))
            self.target_ids.append(torch.tensor(token_ids[i+1:i+max_length+1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [2]:
import tiktoken

def create_dataloader_v1(text, tokenizer_name='gpt2', max_length=256, stride=128, batch_size=32, 
                         shuffle=True, num_workers=0, drop_last=True):
    tokenizer = tiktoken.get_encoding(tokenizer_name)
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader

In [ ]:
with open('the-verdict.txt', 'r', encoding='utf8') as f:
    text = f.read()

# 如果在数据集那里不使用torch.tensor(token_ids[i:i+max_length]),那么会出现打印结果形状(seq_len, bs)
# 而不是我期待的(bs, seq_len)形状
dataloader = create_dataloader_v1(text, batch_size=1, max_length=4, stride=2)
x,y = next(iter(dataloader))
print('x:', x)
print('y:', y)

x: tensor([[ 314, 2993, 1576,  284]])
y: tensor([[2993, 1576,  284, 2666]])


In [14]:
dataloader1 = create_dataloader_v1(text, batch_size=4, max_length=8, stride=4)
data_iter = iter(dataloader1)
x, y = next(data_iter)
print(f'x: {x}')
print(f'y: {y}')

x: tensor([[  843,  8797,   616,  4240,  3432,   262,  2938, 17547],
        [  338,  1613,     0,   198,   198, 27034,    13,   402],
        [  318,    11,   355,   257,  3896,    11,   262,   661],
        [  257,  4922,   339,   550,   262,   976,  3081,   355]])
y: tensor([[ 8797,   616,  4240,  3432,   262,  2938, 17547,   339],
        [ 1613,     0,   198,   198, 27034,    13,   402,   271],
        [   11,   355,   257,  3896,    11,   262,   661,   508],
        [ 4922,   339,   550,   262,   976,  3081,   355,   465]])


## Creating token embeddings

In [15]:
import torch

In [16]:
torch.manual_seed(21)
max_length = 4
embedding_dim = 3
embedddings = torch.nn.Embedding(max_length, embedding_dim)
print(embedddings.weight)

Parameter containing:
tensor([[ 0.1081, -0.4376, -0.7697],
        [-0.1929, -0.3626, -2.8451],
        [ 1.4435,  0.4976,  0.6542],
        [ 0.0754, -1.0767,  0.1269]], requires_grad=True)


In [ ]:
print(embedddings(torch.tensor([2])))
# print(embedddings(2)) 会报错

tensor([[1.4435, 0.4976, 0.6542]], grad_fn=<EmbeddingBackward0>)


## Encoding word position

absolute position embedding

In [3]:
import torch

vocab_size = 1032
embedding_dim = 256
word_embeddings = torch.nn.Embedding(vocab_size, embedding_dim)

In [4]:
max_length = 256

pos_embeddings = torch.nn.Embedding(max_length, embedding_dim)

In [5]:
input_ids = torch.tensor([5, 20, 15, 300])
word_embeds = word_embeddings(input_ids)
print(word_embeds.shape)  # torch.Size([4, 256])

torch.Size([4, 256])


In [6]:
pos_embeds = pos_embeddings(torch.arange(0, input_ids.size(0)))

In [7]:
input_embeds = word_embeds + pos_embeds
print(input_embeds) 

tensor([[-1.6213, -1.7185,  3.8156,  ...,  0.0231,  0.6936,  1.4959],
        [-1.9954,  1.5508, -0.3390,  ..., -0.6037, -1.2367, -0.2314],
        [-2.3114,  0.1415,  2.1014,  ...,  0.7279,  4.0916, -0.0144],
        [-0.8704, -0.3590,  1.1171,  ...,  0.7261, -1.3027,  0.0080]],
       grad_fn=<AddBackward0>)
